# 02 · Exploratory Data Analysis (EDA)

**Goal:** Answer the key business questions through distributions, comparisons, and cross-tabulations.

---
### Business Questions
1. Which categories and sub-categories generate the most sales and profit?
2. Which regions and states are most/least profitable?
3. Which customer segments are most valuable?
4. How does discounting affect profit margin?
5. Which products are loss-makers?

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.visualizations import (
    plot_sales_by_category,
    plot_profit_by_region,
    plot_discount_vs_profit,
    plot_top_n,
    plot_heatmap,
    plotly_treemap,
)

%matplotlib inline
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/superstore_clean.csv', parse_dates=['Order Date', 'Ship Date'])
print(f'Loaded: {df.shape}')

## Q1 · Sales & Profit by Category

In [ ]:
plot_sales_by_category(df, group_col='Category', value_col='Sales',
                       title='Total Sales by Category')
plot_sales_by_category(df, group_col='Category', value_col='Profit',
                       title='Total Profit by Category')

In [ ]:
# Sub-category breakdown
plot_top_n(df, group_col='Sub-Category', value_col='Sales', n=10,
           title='Top 10 Sub-Categories by Sales')
plot_top_n(df, group_col='Sub-Category', value_col='Profit', n=10,
           ascending=True, title='Bottom 10 Sub-Categories by Profit (Loss-makers)')

In [ ]:
# Interactive treemap
fig = plotly_treemap(df, path=['Category', 'Sub-Category'],
                     value_col='Sales', color_col='Profit',
                     title='Sales & Profit Treemap')
fig.show()

## Q2 · Regional & State-Level Profitability

In [ ]:
plot_profit_by_region(df)

In [ ]:
# Top and bottom states
plot_top_n(df, group_col='State', value_col='Profit', n=10,
           title='Top 10 States by Profit')
plot_top_n(df, group_col='State', value_col='Profit', n=10,
           ascending=True, title='Bottom 10 States by Profit')

In [ ]:
# Profit heatmap: Category × Region
plot_heatmap(df, index_col='Category', col_col='Region', value_col='Profit',
             title='Profit by Category × Region')

## Q3 · Customer Segment Analysis

In [ ]:
seg_summary = df.groupby('Segment')[['Sales', 'Profit', 'Quantity']].sum()
seg_summary['Profit Margin (%)'] = (seg_summary['Profit'] / seg_summary['Sales'] * 100).round(2)
print(seg_summary)

seg_summary[['Sales', 'Profit']].plot(kind='bar', figsize=(8, 5), colormap='Set2',
                                        rot=0, title='Sales & Profit by Segment')
plt.ylabel('Amount ($)')
plt.tight_layout()
plt.show()

## Q4 · Discount vs. Profit Margin

In [ ]:
plot_discount_vs_profit(df, hue_col='Category',
                        title='Discount vs. Profit — Does Discounting Hurt Margins?')

In [ ]:
# Correlation between discount and profit
corr = df[['Discount', 'Profit', 'Sales', 'Quantity', 'Profit Margin (%)']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix — Key Metrics')
plt.tight_layout()
plt.show()

## Q5 · Loss-Making Products

In [ ]:
loss_products = (
    df.groupby('Product Name')[['Sales', 'Profit']]
    .sum()
    .query('Profit < 0')
    .sort_values('Profit')
    .head(10)
)
print('Top 10 Loss-Making Products:')
print(loss_products.to_string())